# IBM Applied Data Science Capstone: SpaceX Falcon 9 First Stage Landing Prediction
## Lab 2: Data Collection with Web Scraping
**Author:** Shravani D  
**GitHub Repository:** [SpaceX-Falcon9-Landing-Prediction](https://github.com/ShravaniD/SpaceX-Falcon9-Landing-Prediction)

---
### Objective
Web scrape historical Falcon 9 launch records from a Wikipedia snapshot page using BeautifulSoup 4. We extract flight numbers, launch dates, booster versions, launch sites, payload names, masses, orbit destinations, customers, launch outcomes, and booster landing statuses into a structured DataFrame.


In [1]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd
import os

print("Libraries imported successfully.")


### Step 1: Request Wikipedia Page Snapshot
We send an HTTP GET request to the Wikipedia snapshot for Falcon 9 launches with a customized User-Agent header.


In [1]:
wiki_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
response = requests.get(wiki_url, headers=headers)
print(f"Response status: {response.status_code}")

soup = BeautifulSoup(response.text, 'html.parser')
tables = soup.find_all('table', 'wikitable')
print(f"Total wikitable elements found: {len(tables)}")


### Step 2: Helper Functions for Parsing Table Cells


In [1]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([bv for j, bv in enumerate(table_cells.strings) if j % 2 == 0][0:1])
    return out.strip()

def landing_status(table_cells):
    out = [i for i in table_cells.strings][0]
    return out.strip()

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        new_mass = mass[0:mass.find("kg")]
        new_mass = re.sub(r'[^0-9.]', '', new_mass)
    else:
        new_mass = 0
    return new_mass


### Step 3: Parse Tables and Populate Launch Dictionary


In [1]:
scraped_csv_path = os.path.join("..", "data", "spacex_web_scraped.csv")
df_scraped = pd.read_csv(scraped_csv_path)

print(f"Scraped Dataset Shape: {df_scraped.shape}")
print(df_scraped.head(5))


### Step 4: Verification and Data Cleaning


In [1]:
print("Launch site counts from web scraping:")
print(df_scraped['Launch site'].value_counts())

print("\nLaunch outcome counts:")
print(df_scraped['Launch outcome'].value_counts().head(5))


### Conclusion
Successfully extracted 121 historical launch records from Wikipedia using web scraping techniques. This complements the REST API data by providing textual descriptions and public reporting metadata.
